# Lab 1 · Delta Lake and compute: what the platform gives you

Create a Delta table, change it, read its history, travel back in time, restore it, and watch schema enforcement stop a bad write. Then compare notebook and SQL warehouse compute.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Databricks Intelligence Platform (6% of the exam) · **Time:** about 25 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Part 1 · Set up your lab schema

Every lab uses the `workspace` catalog that Databricks Free Edition gives you, and one schema called `dea_lab`.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dea_lab
COMMENT 'Learning Hub: Data Engineer Associate labs';
USE workspace.dea_lab;
SELECT current_catalog() AS catalog, current_schema() AS schema, current_user() AS me;

## Part 2 · Four commits, one table

**Predict:** how many rows will `DESCRIBE HISTORY` show after these four statements?

In [ ]:
%sql
DROP TABLE IF EXISTS platform_orders;   -- a fresh table, so the version numbers match on every run
CREATE TABLE platform_orders (
  order_id INT, status STRING, amount DECIMAL(10,2));
INSERT INTO platform_orders VALUES (1, 'OPEN', 20.00), (2, 'OPEN', 35.50), (3, 'SHIPPED', 12.00);
UPDATE platform_orders SET status = 'SHIPPED' WHERE order_id = 1;
DELETE FROM platform_orders WHERE order_id = 3;

In [ ]:
%sql
DESCRIBE HISTORY platform_orders;

Each statement is one atomic commit in the table's transaction log. Look at the `operation` and `operationMetrics` columns. Data files are never edited in place: an UPDATE or DELETE writes new files, or a deletion vector that marks removed rows, and commits them atomically.

## Part 3 · Time travel and RESTORE

**Predict:** which rows existed at version 1?

In [ ]:
%sql
SELECT * FROM platform_orders VERSION AS OF 1 ORDER BY order_id;

Someone deleted order 3 by mistake. Put the table back as it was at version 1, then check the history again.

In [ ]:
%sql
RESTORE TABLE platform_orders TO VERSION AS OF 1;
SELECT * FROM platform_orders ORDER BY order_id;

In [ ]:
%sql
DESCRIBE HISTORY platform_orders;

RESTORE is itself a new commit, so you can still see (and undo) everything that happened before it.

## Part 4 · What a Delta table is made of

In [ ]:
%sql
DESCRIBE DETAIL platform_orders;

Find `format` (delta), `location`, `numFiles` and `sizeInBytes`. The location is in storage that Unity Catalog manages for you: this is a **managed** table.

## Part 5 · Schema enforcement

**Predict:** will this append succeed? It has a column the table does not have.

In [ ]:
extra = spark.sql("SELECT 9 AS order_id, 'OPEN' AS status, CAST(5 AS DECIMAL(10,2)) AS amount, 'gift' AS note")
try:
    extra.write.mode("append").saveAsTable("platform_orders")
    print("Append succeeded")
except Exception as e:
    print("Blocked by schema enforcement:", type(e).__name__)
    print(str(e).splitlines()[0][:300])

Delta rejects writes that do not match the table schema. When the new column is intended, opt in to **schema evolution** explicitly:

In [ ]:
extra.write.mode("append").option("mergeSchema", "true").saveAsTable("platform_orders")
display(spark.table("platform_orders").orderBy("order_id"))

## Part 6 · Where did that run?

In [ ]:
print("Spark version:", spark.version)
print("Rows:", spark.table("platform_orders").count())

This notebook ran on **serverless notebook compute**: no cluster to size or start. Now open **SQL Editor**, pick the **Serverless Starter Warehouse**, and run:

```text
SELECT status, count(*) AS orders, sum(amount) AS revenue
FROM workspace.dea_lab.platform_orders
GROUP BY status;
```

A SQL warehouse runs SQL only. The Python cell in Part 5 could not run there, which is why a PySpark task in a job needs notebook (cluster or serverless) compute, not a warehouse.

## Expected results

```text
Part 2: DESCRIBE HISTORY shows 4 versions (0 CREATE TABLE, 1 WRITE, 2 UPDATE, 3 DELETE).
Part 3: version 1 has orders 1, 2 and 3, all with their original status. After RESTORE the
        table has 3 rows again and the history gains version 4, operation RESTORE.
Part 4: format = delta; numFiles is small (1-3); location is a managed storage path.
Part 5: the first append is blocked (a schema mismatch error); with mergeSchema it succeeds,
        the table gains a "note" column, and older rows show NULL for it.
Part 6: the notebook and the SQL warehouse return the same totals from the same table.
```

## Exam takeaways

- Delta Lake adds ACID commits, history, time travel and schema enforcement. It does not add multi-language notebooks, alerts or job scheduling.
- A SQL warehouse is the answer for many analysts running SQL and BI; it never runs PySpark.
- RESTORE and time travel only reach versions whose data files have not been removed by VACUUM.